In [0]:
from pyspark.sql.functions import current_timestamp, col, trim, concat, lit, count, collect_set, when, length
from functools import reduce

# Define Target catalog and bronze layer, and schema for silver layer
TARGET_CATALOG = "workspace"
BRONZE_SCHEMA = "mimic_iv_dataset_bronze"
SILVER_SCHEMA = "mimic_iv_dataset_silver"

# For reordering delta table columns
DELTA_TABLE_METADATA_COLUMNS = ["_ingestion_timestamp", "_cleaned_timestamp", "_source_file"]

# Create schema
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {TARGET_CATALOG}.{SILVER_SCHEMA}")

In [0]:
# d_icd_diagnoses
# 1) There should not be any missing data in this table
# 2) There should not be extra whitespaces in the data
# 3) ICD Code and ICD Version should be unique

# Read from bronze layer
d_icd_diagnoses_bronze_df = spark.table(f"{TARGET_CATALOG}.{BRONZE_SCHEMA}.d_icd_diagnoses")

# Silver layer transformations
d_icd_diagnoses_silver_df = (
    d_icd_diagnoses_bronze_df.filter(
        reduce(lambda a, b: a & b, [
            col(feat_name).isNotNull() for feat_name in d_icd_diagnoses_bronze_df.columns
        ])
    ) # Filters out any missing data (Condition 1)
    .select([
        trim(col(feat_name)).alias(feat_name) if data_type == "string" else col(feat_name)
        for feat_name, data_type in d_icd_diagnoses_bronze_df.dtypes
    ]) # Trims whitespaces (Condition 2)
    .dropDuplicates(["icd_code", "icd_version"]) # Removes duplicates (Condition 3)
    .withColumn("_cleaned_timestamp", current_timestamp()) # Add timestamp
)

# Reordering columns
d_icd_diagnoses_silver_df = d_icd_diagnoses_silver_df.select(
    [feat_name 
     for feat_name in d_icd_diagnoses_silver_df.columns if feat_name[0] != "_"] +
    DELTA_TABLE_METADATA_COLUMNS)

# Write to silver layer
(d_icd_diagnoses_silver_df.write.format("delta")
                                .mode("overwrite")
                                .option("overwriteSchema", True)
                                .saveAsTable(f"{TARGET_CATALOG}.{SILVER_SCHEMA}.d_icd_diagnoses"))

print("Silver layer (d_icd_diagnoses): success")

In [0]:
# diagnoses_icd
# 1) There should not be any missing data in this table
# 2) There should not be extra whitespaces in the data
# 3) subject_id, hadm_id should be string not integer in the table
# 4) ICD Code and ICD Version should match what is seen under d_icd_diagnoses

# Get the ICD Code and ICD Version pairs from d_icd_diagnoses as a list of "code|version" pair
valid_icd_pairs = [
    f"{row.icd_code}|{row.icd_version}"
    for row in d_icd_diagnoses_silver_df.select("icd_code", "icd_version")
                                        .distinct().collect()
]

# Read from bronze layer
diagnoses_icd_bronze_df = spark.table(f"{TARGET_CATALOG}.{BRONZE_SCHEMA}.diagnoses_icd")

# Silver layer transformations
diagnoses_icd_silver_df = (
    diagnoses_icd_bronze_df.filter(
        reduce(lambda a, b: a & b, [
            col(column).isNotNull() for column in diagnoses_icd_bronze_df.columns
        ])
    ) # Filters out any missing data (Condition 1)
    .select([
        trim(col(feat_name)).alias(feat_name) if data_type == "string" else col(feat_name)
        for feat_name, data_type in diagnoses_icd_bronze_df.dtypes
    ]) # Trims whitespaces (Condition 2)
    .withColumn("subject_id", col("subject_id").cast("string")) # Convert data type (Condition 3)
    .withColumn("hadm_id", col("hadm_id").cast("string"))
    .filter(concat(col("icd_code"), lit("|"), col("icd_version")).isin(valid_icd_pairs)) # Keeps only rows whose ICD pair exists in d_icd_diagnoses (Condition 4)
    .withColumn("_cleaned_timestamp", current_timestamp()) # Add timestamp
)

# Reordering columns
diagnoses_icd_silver_df = diagnoses_icd_silver_df.select(
    [feat_name 
     for feat_name in diagnoses_icd_silver_df.columns if feat_name[0] != "_"] +
    DELTA_TABLE_METADATA_COLUMNS)

# Write to silver layer
(diagnoses_icd_silver_df.write.format("delta")
                                .mode("overwrite")
                                .option("overwriteSchema", True)
                                .saveAsTable(f"{TARGET_CATALOG}.{SILVER_SCHEMA}.diagnoses_icd"))

print("Silver layer (diagnoses_icd): success")

In [0]:
# drgcodes
# 1) There should not be extra whitespaces in the data
# 2) subject_id, hadm_id should be string not integer in the table
# 3) The patient ID, hospital ID and drug identifiers (type and code) should be unique
# 4) The drug details (type, code, description) should be unique. Note that this was not the case in the data.
#    - We would have to create a separate table to fix this

# Read from bronze layer
drgcodes_bronze_df = spark.table(f"{TARGET_CATALOG}.{BRONZE_SCHEMA}.drgcodes")

# Group by drg_type and drg_code, collect unique descriptions
drg_conflicts = (
    drgcodes_bronze_df
    .select("drg_type", "drg_code", "description")
    .distinct()
    .groupBy("drg_type", "drg_code")
    .agg(
        collect_set("description").alias("descriptions"),
        count("description").alias("description_count")
    )
    .filter(col("description_count") > 1)
    .orderBy("drg_type", "drg_code")
)

conflict_count = drg_conflicts.count()

try:
    if conflict_count > 0:
        raise ValueError(f"Found {conflict_count} DRG type(s) and code(s) with multiple descriptions")
except ValueError as ve:
    print(ve)
    print("Please review and fix multiple descriptions manually")
    drg_conflicts.show(truncate = False)

In [0]:
# d_drgcodes
# Contains unique combinations of drg_type, drg_code, and description
# 1) There should not be extra whitespaces in the data
# 2) The drug details (type, code, description) should be unique

d_drgcodes_cols = ["drg_type", "drg_code", "description", "_source_file"]

# Create silver layer d_drgcodes with unique drug details
d_drgcodes_silver_df = (
    drgcodes_bronze_df.select(d_drgcodes_cols)
    .select([
        trim(col(feat_name)).alias(feat_name) if data_type == "string" else col(feat_name)
        for feat_name, data_type in drgcodes_bronze_df.select(d_drgcodes_cols).dtypes
    ]) # Trim whitespaces (Condition 1)
)

# Fixing unique drug details manually (Condition 2)
# Note: This is based on the drug type and drug code found earlier.
#       We will be editing the incorrect descriptions manually.

# Create mapped descriptions for overwriting and update it
drgcodes_updated_desc = {
    ("HCFA", 65): "INTRACRANIAL HEMORRHAGE OR CEREBRAL INFARCTION W CC OR TPA IN 24 HRS",
    ("HCFA", 441): "DISORDERS OF LIVER EXCEPT MALIG, CIRR, ALC HEPA W MCC",
    ("HCFA", 871): "SEPTICEMIA OR SEVERE SEPSIS W/O MV >96 HOURS W MCC"
}

description_expr = col("description")
for (drg_type, drg_code), updated_desc in drgcodes_updated_desc.items():
    description_expr = when(
        (col("drg_type") == drg_type) & (col("drg_code") == drg_code),
        updated_desc
    ).otherwise(description_expr)

d_drgcodes_silver_df = d_drgcodes_silver_df.withColumn("description", description_expr)

# After manual fixes, get unique combinations
d_drgcodes_silver_df = (
    d_drgcodes_silver_df
    .distinct() # Get unique combinations
    .withColumn("_cleaned_timestamp", current_timestamp()) # Add timestamp
)

# Verifying update fix
display(d_drgcodes_silver_df.filter(
        (col("drg_type") == "HCFA") & 
        ((col("drg_code") == 65) | (col("drg_code") == 441) | (col("drg_code") == 871))
))

# Reordering columns
d_drgcodes_silver_df = d_drgcodes_silver_df.select(
    [feat_name 
     for feat_name in d_drgcodes_silver_df.columns if feat_name[0] != "_"] +
    ["_cleaned_timestamp", "_source_file"])

# Write to silver layer
(d_drgcodes_silver_df.write.format("delta")
                            .mode("overwrite")
                            .option("overwriteSchema", True)
                            .saveAsTable(f"{TARGET_CATALOG}.{SILVER_SCHEMA}.d_drgcodes"))

print("Silver layer (d_drgcodes): success")

In [0]:
# drgcodes
# 1) There should not be extra whitespaces in the data
# 2) subject_id, hadm_id should be string not integer in the table
# 3) The patient ID, hospital ID and drug identifiers (type and code) should be unique
# 4) Remove description since it is under d_drgcodes now

# Silver layer transformations
drgcodes_silver_df = (
    drgcodes_bronze_df.drop("description") # Remove description column (Condition 4)
    .select([
        trim(col(feat_name)).alias(feat_name) if data_type == "string" else col(feat_name)
        for feat_name, data_type in drgcodes_bronze_df.drop("description").dtypes
    ]) # Trims whitespaces (Condition 1)
    .withColumn("subject_id", col("subject_id").cast("string")) # Convert data type (Condition 2)
    .withColumn("hadm_id", col("hadm_id").cast("string"))
    .dropDuplicates(["subject_id", "hadm_id", "drg_type", "drg_code"]) # Removes duplicates (Condition 3)
    .withColumn("_cleaned_timestamp", current_timestamp()) # Add timestamp
)

# Reordering columns
drgcodes_silver_df = drgcodes_silver_df.select(
    [feat_name 
     for feat_name in drgcodes_silver_df.columns if feat_name[0] != "_"] +
    DELTA_TABLE_METADATA_COLUMNS)

# Write to silver layer
(drgcodes_silver_df.write.format("delta")
                            .mode("overwrite")
                            .option("overwriteSchema", True)
                            .saveAsTable(f"{TARGET_CATALOG}.{SILVER_SCHEMA}.drgcodes"))

print("Silver layer (drgcodes): success")

In [0]:
# d_hcpcs
# 1) There should not be extra whitespaces in the data
# 2) Long and short description are confusing, and majority of the data does not include a long description, while having long phrases/sentences written in the short description.
#    Instead of having two descriptions, we will combine them into one description while removing redundant information. This will also resolve the majority of missing data in long description (Note: All data have short description but not all data have long description)
#    From analysing the descriptions, data with both long and short descriptions are similar, therefore we will just take the description that has the most information as the final description column.
#    Assumption: If both descriptions have the same amount of characters, we are assuming the long description is more informative than the short description
# 3) The hcpcs details (code and category) should be unique.

# Read from bronze layer
d_hcpcs_bronze_df = spark.table(f"{TARGET_CATALOG}.{BRONZE_SCHEMA}.d_hcpcs")

# Silver layer transformations
d_hcpcs_silver_df = (
    d_hcpcs_bronze_df
    .select([
        trim(col(feat_name)).alias(feat_name) if data_type == "string" else col(feat_name)
        for feat_name, data_type in d_hcpcs_bronze_df.dtypes
    ]) # Trims whitespaces (Condition 1)
    .withColumn(
        "description",
        when(
            col("long_description").isNotNull() &
            (length(col("long_description")) >= length(col("short_description"))),
            col("long_description")
        ).otherwise(col("short_description"))
    ) # Create single description column based on longer text from long_description vs short_description (Condition 2)
    .drop("long_description", "short_description") # Drop original description columns
    .dropDuplicates(["code", "category"]) # Remove duplicates (Condition 3)
    .withColumn("_cleaned_timestamp", current_timestamp()) # Add timestamp
)

# Reordering columns
d_hcpcs_silver_df = d_hcpcs_silver_df.select(
    [feat_name 
     for feat_name in d_hcpcs_silver_df.columns if feat_name[0] != "_"] +
    DELTA_TABLE_METADATA_COLUMNS)

# Write to silver layer
(d_hcpcs_silver_df.write.format("delta")
                        .mode("overwrite")
                        .option("overwriteSchema", True)
                        .saveAsTable(f"{TARGET_CATALOG}.{SILVER_SCHEMA}.d_hcpcs"))

print("Silver layer (d_hcpcs): success")

In [0]:
# hcpcsevents
# 1) There should not be extra whitespaces in the data
# 2) subject_id, hadm_id should be string not integer in the table
# 3) short description is not a good column name, use description instead for standardisation

# Read from bronze layer
hcpcsevents_bronze_df = spark.table(f"{TARGET_CATALOG}.{BRONZE_SCHEMA}.hcpcsevents")

# Silver layer transformations
hcpcsevents_silver_df = (
    hcpcsevents_bronze_df
    .select([
        trim(col(feat_name)).alias(feat_name) if data_type == "string" else col(feat_name)
        for feat_name, data_type in hcpcsevents_bronze_df.dtypes
    ]) # Trims whitespaces (Condition 1)
    .withColumn("subject_id", col("subject_id").cast("string")) # Convert data type (Condition 2)
    .withColumn("hadm_id", col("hadm_id").cast("string"))
    .withColumnRenamed("short_description", "description") # Rename column (Condition 3)
    .withColumn("_cleaned_timestamp", current_timestamp()) # Add timestamp
)

# Reordering columns
hcpcsevents_silver_df = hcpcsevents_silver_df.select(
    [feat_name 
     for feat_name in hcpcsevents_silver_df.columns if feat_name[0] != "_"] +
    DELTA_TABLE_METADATA_COLUMNS)

# Write to silver layer
(hcpcsevents_silver_df.write.format("delta")
                            .mode("overwrite")
                            .option("overwriteSchema", True)
                            .saveAsTable(f"{TARGET_CATALOG}.{SILVER_SCHEMA}.hcpcsevents"))

print("Silver layer (hcpcsevents): success")

In [0]:
# Verify all delta tables in silver layer exists through displaying samples, dynamically
table_names = [table.name for table in spark.catalog.listTables(f"{TARGET_CATALOG}.{SILVER_SCHEMA}")]

for table_name in table_names:
    print(f"Delta table: {table_name}")
    
    df = spark.sql(f"SELECT * FROM {TARGET_CATALOG}.{SILVER_SCHEMA}.{table_name} LIMIT 5")
    display(df)

    print("")